# Preparation

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

import random
from ollama import chat

NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "X"
OLLAMA_MODEL = "qwen3-embedding:4b"
INDEX_NAME = "documents"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [ ]:
with driver.session() as session:
    session.run(f"DROP INDEX {INDEX_NAME} IF EXISTS")

    session.run(f"""
        CREATE VECTOR INDEX {INDEX_NAME} IF NOT EXISTS
        FOR (p:product) ON (p.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: 2560,
            `vector.similarity_function`: 'cosine'
          }}
        }}
    """)

In [6]:
ollama_embedder = OllamaEmbeddings(model=OLLAMA_MODEL)

retriever = VectorRetriever(
    driver=driver,
    index_name=INDEX_NAME,
    embedder=ollama_embedder,
    return_properties=["name", "document"],
)

# Suchanfrage definieren
query = "I would like a toy for my cat!"

# Suche durchführen (Top k ähnlichste Ergebnisse)
search_results = retriever.search(query_text=query, top_k=5)

print(f"Suchanfrage: '{query}'\n")
print("Ergebnisse:")
for i, result in enumerate(search_results.items, 1):
    print(i, result)
    print(f"[{i}] Score: {result.metadata.get('score'):.4f}")

Suchanfrage: 'I would like a toy for my cat!'

Ergebnisse:
1 content='{\'name\': \'NAVAJO BUNGEE CAT TEASR 27IN\', \'document\': \'- product: NAVAJO BUNGEE CAT TEASR 27IN\\n- reviews: \\n#1:\\nsummary: Not the one!\\ntext: "This might be okay for kittens that are more prone to entertain themselves but my 6 yr old cats were not interested, even when I tried to turn it into an interactive toy. All I can say it that it seems to be constructed well.  Otherwise another boring cat toy!"\\n\'}' metadata={'score': 0.8781543970108032, 'nodeLabels': ['product'], 'id': '4:da4dd44f-bbf2-4f39-b5b3-3529e7224533:16463'}
[1] Score: 0.8782
2 content='{\'name\': \'Leegoal New Small Dog Cat Pet Stripe Bow Tie Neck Tie White Collar Choose Color\', \'document\': \'- product: Leegoal New Small Dog Cat Pet Stripe Bow Tie Neck Tie White Collar Choose Color\\n- reviews: \\n#1:\\nsummary: looks great in this costume\\ntext: "My cat, Senny, looks great in this costume."\\n\'}' metadata={'score': 0.84856736660003

In [28]:
q = """
MATCH (b:brand)-[:has_brand]-(p:product)-[:has_category]-(cat:category)
WITH b, cat, count(DISTINCT p) AS answer_count, collect(p.id) AS ground_truth
WHERE answer_count >= 1 AND answer_count <= 15
RETURN b.name AS brand_name, cat.name AS category_name, answer_count, ground_truth
"""

q = """
MATCH (cat1:category)-[:has_category]-(:product)-[:also_buy]-(target:product)-[:has_color]-(col1:color)
WITH cat1, col1, count(DISTINCT target) AS answer_count, collect(DISTINCT target.id) AS ground_truth
WHERE answer_count >= 1 AND answer_count <= 15
RETURN cat1.name, col1.name, answer_count, ground_truth
"""

records, summary, keys = driver.execute_query(
    q,
)

for record in random.sample(records, 10):
    print(record)


print(f"returned {len(records)} records in {summary.result_available_after} ms.")

<Record cat1.name='cooking utensils' col1.name='brick' answer_count=1 ground_truth=['249029']>
<Record cat1.name='kayaking' col1.name='galaxy' answer_count=3 ground_truth=['639797', '639787', '639805']>
<Record cat1.name='made in usa' col1.name='red/navy' answer_count=2 ground_truth=['573542', '111564']>
<Record cat1.name='water bottle cages' col1.name='coyote' answer_count=1 ground_truth=['393028']>
<Record cat1.name='paintball' col1.name='lime green' answer_count=4 ground_truth=['848206', '324170', '324168', '177860']>
<Record cat1.name='scooters' col1.name='cyan blue' answer_count=1 ground_truth=['64226']>
<Record cat1.name='vibration dampeners' col1.name='green/black' answer_count=1 ground_truth=['114464']>
<Record cat1.name='mat bags' col1.name='ivory' answer_count=1 ground_truth=['849241']>
<Record cat1.name='sport watches' col1.name='pink' answer_count=11 ground_truth=['703147', '601730', '489941', '382295', '381345', '123156', '111952', '47830', '3540', '3536', '676']>
<Record 

In [33]:
prompt = """You are an intelligent assistant that generates queries about Amazon items for a QA dataset.
I will provide you with a golden relational path from an Amazon product recommendation knowledge graph, which leads to one or more target products.
Your task is to create a natural-sounding customer query that leads to the target products as the answer.

Example:
Path: ('SUNVP':brand)-[:has_brand]-(target:product)-[:has_category]-('backpacking packs':category)
Query: What are some good backpacking packs from the brand SUNVP?

Path: ('vibration dampeners':category)-[:has_category]-(:product)-[:also_buy]-(target:product)-[:has_color]-('galaxy':color)
Query:"""

response = chat(
    model="gemma4:26b",
    messages=[
        {
            "role": "user",
            "content": prompt,
        },
    ],
)
print(response.message.content)

What galaxy colored products are often bought along with vibration dampeners?
